# GPU PDE Lab — run it without installing anything

This notebook runs the project's numerics on a free Colab GPU: no Nix, no Qt, no local setup.

**Set the runtime to a GPU first** — *Runtime → Change runtime type → T4 GPU*. Everything
below still runs without one; the GPU-specific cells will say so and skip rather than fail.

What it covers:

1. Does the discretization converge at the order the theory predicts?
2. Why is the GPU *slower* than the CPU on small grids?
3. How much of the GPU is the obvious implementation actually using — and what does
   a hand-written fused kernel recover?

The third is the interesting one. Short answer on the reference card: the eager
implementation reaches about 7% of achievable memory bandwidth, and a fused Triton
kernel reaches 93%.


## Setup


In [ ]:
!git clone --depth 1 https://github.com/LuxKaiwalker/gpu-pde-lab.git
%cd gpu-pde-lab
# The solver core needs only numpy and torch, both already present on Colab.
# `[bench]` adds matplotlib for the figures. PySide6 is deliberately not pulled
# in: the desktop app is a separate extra and this notebook does not need it.
!pip install -q -e '.[bench]'


In [ ]:
import torch
from gpu_pde_lab.core.backend import cuda_is_usable, cuda_free_memory_bytes

print('torch           ', torch.__version__)
print('CUDA usable     ', cuda_is_usable())
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    peak = p.memory_clock_rate * 1e3 * 2 * p.memory_bus_width / 8
    print('device          ', p.name)
    print('peak bandwidth  ', f'{peak/1e9:.0f} GB/s')
    print('free memory     ', f'{cuda_free_memory_bytes()/1024**2:.0f} MB')
else:
    print('No GPU attached — Runtime > Change runtime type > T4 GPU')


## 1. Does it converge at order 2?

The convergence study uses an initial condition that is an eigenfunction of the discrete
Laplacian, so an exact closed-form solution exists to measure error against:

$$u_0 = \sin(\pi x/L)\sin(\pi y/L)  \Rightarrow  u(x,y,t) = e^{-2\alpha\pi^2 t/L^2}\,u_0$$

A 5-point stencil is second-order accurate in space, so halving $h$ should quarter the error.
This is the check that the solver computes the right thing at all — everything about
performance below is only interesting if this passes.


In [ ]:
from gpu_pde_lab.benchmarking.convergence import spatial_convergence

result = spatial_convergence([17, 33, 65, 129, 257])
print(f"{'h':>12} {'Linf error':>14} {'observed order':>16}")
orders = [None] + list(result.observed_orders)
for point, order in zip(result.points, orders):
    shown = f'{order:.3f}' if order is not None else '-'
    print(f'{point.h_or_dt:>12.5f} {point.error_linf:>14.3e} {shown:>16}')

avg = sum(result.observed_orders) / len(result.observed_orders)
print(f'\nmean observed order: {avg:.3f}   (theory: 2)')


## 2. Where does the GPU actually win?

Not everywhere. A GPU kernel launch costs the host a few microseconds whether the kernel
does any work or not. Below a few hundred cells on a side there is not enough work to
amortize that, and the CPU wins outright.

This is the single most useful thing to know before putting a GPU in anything.


In [ ]:
from gpu_pde_lab.benchmarking.suite import run_cpu_gpu_sweep
from gpu_pde_lab.core.backend import cuda_is_usable

if not cuda_is_usable():
    print('No usable GPU — skipping (this cell compares CPU against GPU).')
else:
    sweep = run_cpu_gpu_sweep([64, 128, 256, 512, 1024, 2048], n_steps=30)
    print(f"{'N':>6} {'CPU ms':>10} {'GPU ms':>10} {'speedup':>9}")
    for n, c, g, s in zip(sweep.sizes, sweep.cpu, sweep.gpu, sweep.speedups()):
        flag = '  <- GPU loses' if s < 1 else ''
        print(f'{n:>6} {c.seconds_per_step*1e3:>10.3f} {g.seconds_per_step*1e3:>10.3f} {s:>8.2f}x{flag}')


## 3. The real question: how much of the GPU is being used?

A 5-point stencil moves at least 8 bytes per cell (one fp32 read, one write) and does about
6 flops. That arithmetic intensity is far below any modern GPU's ridge point, so the only
thing that matters is bandwidth — and the obvious implementation wastes most of it.

Writing the update as shifted-slice arithmetic means every intermediate is a full-sized
tensor with its own round trip to memory. Measured at the dispatch layer that is about
**33 array passes** where 2 would do, spread over **17 kernel launches**.

Four implementations of the identical update, compared:

| | what it is | kernels/step |
|---|---|---|
| `baseline` | eager shifted slices | 17 |
| `cuda_graph` | the same ops, captured and replayed as one submission | 18 |
| `compiled` | `torch.compile` — inductor fuses the elementwise chain | 4 |
| `triton` | hand-written fused kernel, boundary folded in | **1** |

`cuda_graph` removes launch overhead but moves exactly as much memory; `triton` removes the
memory traffic. Keeping both separates two bottlenecks that a wall-clock number alone cannot.


In [ ]:
from gpu_pde_lab.kernels.registry import available_implementations

for d in available_implementations():
    status = 'ok' if d.available else f'unavailable — {d.unavailable_reason}'
    print(f'{d.name:<12} {status}')
    print(f'{"":<12} {d.description}')


In [ ]:
from gpu_pde_lab.benchmarking.roofline import run_roofline_sweep
from gpu_pde_lab.core.backend import cuda_is_usable

if not cuda_is_usable():
    print('No usable GPU — skipping the roofline sweep.')
    sweep = None
else:
    # Keep it modest so the notebook stays quick; the committed results in
    # docs/results/ go to N=8192.
    sweep = run_roofline_sweep([256, 512, 1024, 2048], n_steps=30)
    print(sweep.device.summary(), '\n')
    hdr = f"{'impl':<12} {'N':>6} {'ms/step':>10} {'GB/s':>8} {'%memcpy':>9} {'err':>10}"
    print(hdr); print('-'*len(hdr))
    for m in sweep.measurements:
        print(f'{m.impl:<12} {m.n:>6} {m.seconds_per_step*1e3:>10.4f} '
              f'{m.achieved_bandwidth_bytes_per_s/1e9:>8.1f} '
              f'{m.fraction_of_achievable*100:>8.1f}% {m.max_abs_diff_from_baseline:>10.1e}')


### Plot it against the roofline

Two ceilings are drawn. The dotted line is the card's spec-sheet peak; the dashed line is a
measured device-to-device `memcpy` at the same size, which is the honest ceiling for a
read-once-write-once kernel. A fused stencil cannot beat a memcpy.


In [ ]:
import matplotlib.pyplot as plt

if sweep is None:
    print('Nothing measured — run the cell above on a GPU runtime.')
else:
    fig, (a, b) = plt.subplots(1, 2, figsize=(13, 4.5))
    for impl in sweep.impls_present():
        ms = sorted(sweep.by_impl(impl), key=lambda m: m.n)
        a.plot([m.n for m in ms], [m.achieved_bandwidth_bytes_per_s/1e9 for m in ms], 'o-', label=impl)
        b.plot([m.n for m in ms], [m.fraction_of_achievable*100 for m in ms], 'o-', label=impl)
    ns = sorted({m.n for m in sweep.measurements})
    ceil = [next(m.copy_ceiling_bytes_per_s/1e9 for m in sweep.measurements if m.n == n) for n in ns]
    a.plot(ns, ceil, 'k--', label='memcpy ceiling')
    a.axhline(sweep.measurements[0].peak_bandwidth_bytes_per_s/1e9, color='k', ls=':', label='spec peak')
    b.axhline(100, color='k', ls='--')
    for ax, yl, t in ((a, 'achieved useful GB/s', 'Bandwidth vs the roofline'),
                      (b, '% of memcpy ceiling', 'Fraction of achievable')):
        ax.set_xscale('log', base=2); ax.set_xlabel('N (grid is N x N)')
        ax.set_ylabel(yl); ax.set_title(t); ax.legend(fontsize=8); ax.grid(True, which='both', alpha=0.3)
    plt.tight_layout(); plt.show()


## 4. Watch a scheme go unstable

Explicit FTCS is only *conditionally* stable: von Neumann analysis gives
$\Delta t \le h^2/(4\alpha)$. Past that bound the highest frequency the grid can represent —
the cell-to-cell checkerboard — is amplified every step.

One subtlety worth seeing: a smooth initial condition contains almost none of that mode, so
an over-the-bound run looks fine for a while. There is nothing at the unstable frequency to
amplify until round-off seeds it. The cell below seeds it explicitly.

Running this on a faster GPU does not fix it. It reaches `inf` sooner.


In [ ]:
import numpy as np
from gpu_pde_lab.core.backend import Backend, BackendSpec
from gpu_pde_lab.core.boundary import BoundaryKind
from gpu_pde_lab.core.grid import Grid2D
from gpu_pde_lab.pdes.heat import HeatEquation

grid, backend = Grid2D(nx=128, ny=128), Backend(BackendSpec('numpy', 'cpu', 'fp64'))

def amplitude_history(factor, steps=220, seed=False):
    pde = HeatEquation(grid, backend, BoundaryKind.NEUMANN, alpha=0.1)
    pde.reset(ic='multiple_sources')
    if seed:
        rng = np.random.default_rng(0)
        u0 = pde.primary_field_numpy()
        pde.reset_from_array(u0 + 1e-6 * rng.standard_normal(u0.shape))
    dt = factor * pde.dt_max()
    out = []
    for _ in range(steps):
        pde.step(dt)
        u = pde.primary_field_numpy()
        out.append(float(np.max(np.abs(u))) if np.all(np.isfinite(u)) else np.nan)
    return out

plt.figure(figsize=(7, 4.5))
plt.semilogy(amplitude_history(1.10, seed=True), label='1.10 x dt_max (unstable)')
plt.semilogy(amplitude_history(0.40), label='0.40 x dt_max (stable)')
plt.xlabel('step'); plt.ylabel('max |u|'); plt.legend(); plt.grid(True, which='both', alpha=0.3)
plt.title('Past the stability bound the grid-scale mode grows ~20% per step')
plt.tight_layout(); plt.show()


## Where to go next

- `doc.md` — the full technical reference: discretizations, stability derivations, architecture
- `docs/results/roofline.md` — committed measurements on an RTX 2080 Ti, with the raw CSV
- `src/gpu_pde_lab/kernels/triton_stencil.py` — the fused kernel, and how the boundary
  condition is folded into a single pass without a second launch
- `python -m gpu_pde_lab.app` — the desktop lab, twelve interactive experiments
  (needs `pip install '.[lab]'` and a display)
